<a href="https://colab.research.google.com/github/Adrian-HI-LO/Adrian-HI-LO/blob/main/Coneccion_AWS_Bussines_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalar dependencias.

In [ ]:
# Instalamos las versiones estables de SageMaker y Boto3
!pip install sagemaker==2.215.0 boto3 --quiet
print("Instalaciones completadas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/5

In [ ]:
## Cargar credenciales y sesiones

In [ ]:
from google.colab import userdata
import sagemaker
import boto3

# 1. Cargar credenciales desde los Secrets
aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY')
aws_region = 'us-east-1'

# 2. Iniciar sesión en AWS
boto_session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

# 3. Datos exactos de tu arquitectura
role = 'arn:aws:iam::231351515337:role/Rol-Entrenamiento-SageMaker'
ruta_modelo_s3 = 's3://sagemaker-us-east-1-231351515337/sagemaker/tarjetas-credito-xgboost/output/model.tar.gz'

print("Sesión iniciada correctamente. Listo para conectar con el modelo en S3.")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
Sesión iniciada correctamente. Listo para conectar con el modelo en S3.


## Encender la API

In [ ]:
from sagemaker.image_uris import retrieve

# Nuestro nombre fijo y predecible
nombre_mi_api = 'api-tarjetas-xgboost-v1'

container = retrieve('xgboost', aws_region, '1.5-1')

xgb_model = sagemaker.model.Model(
    image_uri=container,
    model_data=ruta_modelo_s3,
    role=role,
    sagemaker_session=sagemaker_session
)

print(f"Encendiendo el servidor con el nombre fijo: {nombre_mi_api}...")
print("A esperar los guiones (---)...")

try:
    predictor = xgb_model.deploy(
        initial_instance_count=1,
        instance_type='ml.t2.medium',
        endpoint_name=nombre_mi_api # <--- FORZAMOS EL NOMBRE AQUÍ
    )
    print("\n¡Despliegue terminado por el SDK!")
except Exception as e:
    print(f"\nEl SDK arrojó una advertencia, pero revisaremos si AWS lo encendió: {e}")

print(f"Para verificar en AWS, busca el Endpoint llamado: {nombre_mi_api}")

Encendiendo el servidor con el nombre fijo: api-tarjetas-xgboost-v1...
A esperar los guiones (---)...
-------------!
¡Despliegue terminado por el SDK!
Para verificar en AWS, busca el Endpoint llamado: api-tarjetas-xgboost-v1


## Apagar la API

In [ ]:
import boto3

nombre_mi_api = 'api-tarjetas-xgboost-v1'
print(f"Apagando el servidor fijo: {nombre_mi_api}...")

sm_client = boto_session.client('sagemaker', region_name=aws_region)

try:
    sm_client.delete_endpoint(EndpointName=nombre_mi_api)
    sm_client.delete_endpoint_config(EndpointConfigName=nombre_mi_api)
    print("✅ Endpoint apagado y eliminado de forma segura.")
except Exception as e:
    print(f"Aviso: {e} (Probablemente ya estaba apagado).")

Apagando el servidor fijo: api-tarjetas-xgboost-v1...
✅ Endpoint apagado y eliminado de forma segura.


# Usar en Caso de Emergencia

In [ ]:
import boto3

print("Escaneando AWS en busca de Endpoints encendidos...")
# Nos conectamos directo al motor de AWS
sm_client = boto_session.client('sagemaker', region_name=aws_region)

# Pedimos la lista de todos los servidores que estén vivos
endpoints_vivos = sm_client.list_endpoints(StatusEquals='InService')['Endpoints']

if len(endpoints_vivos) == 0:
    print("✅ No hay APIs encendidas. Estás a salvo de cobros.")
else:
    for ep in endpoints_vivos:
        nombre_perdido = ep['EndpointName']
        print(f"¡Encontré un servidor huérfano!: {nombre_perdido}")
        print("Destruyéndolo...")

        # Lo borramos a la fuerza
        sm_client.delete_endpoint(EndpointName=nombre_perdido)
        sm_client.delete_endpoint_config(EndpointConfigName=nombre_perdido)

        print(f"✅ Servidor '{nombre_perdido}' apagado y eliminado exitosamente.")

Escaneando AWS en busca de Endpoints encendidos...
✅ No hay APIs encendidas. Estás a salvo de cobros.
